# 5-3: Sentiment Analysis in Python

In this tutorial, we'll learn how to compare two approaches to sentiment analysis using the dataset of New York Times articles about the 2020 presidential election we compiled in Week 4 (see `4-1_api_lessons.ipynb`). This dataset––`election2020_articles.csv`––should be in your data folder. If it's not there, check the course repository and/or revisit the API lesson to rebuild the dataset.

Sentiment analysis is a computational method for identifying and classifying the emotional tone or attitude expressed in text, often as positive, negative, neutral, or more specific emotions. In DH, it can help scholars trace changes in public opinion, compare emotional patterns across literary works, analyze historical newspapers or archives, and study how communities represent people, events, or ideas over time.

Sentiment analysis can be fuzzy. Sentiments involving sarcasm, irony, ambiguity, historical language, and cultural context can be hard to tease out. That's true with computational methods as much as it is for human readers. But with sentiment analysis, the challenge primarily lies in matching the method to the sentiment and the data. Because emotional meaning depends heavily on genre, audience, and interpretation, its results should often be treated as starting points for humanistic analysis, especially at the scale of individual examples.

The first approach we'll look at is __rule-based sentiment analysis__ with VADER. This method uses a built-in sentiment lexicon (list of words) and a set of rules to score text. This is a straightforward method of sentiment analysis, but it's brittle because it ignores the context of words.

The second approach is __machine-learning sentiment analysis__ with `scikit-learn`. This method learns patterns from labeled examples and attempts to identify sentiment based on those labels. It's much more flexible, but more complicated, as we'll see.

Our workflow will look like this:

1. Load and inspect the election article dataset
2. Build a text column from each article's headline and lead paragraph
3. Use VADER to assign rule-based sentiment scores and labels
4. Use the VADER labels as weak labels for a supervised machine-learning exercise
5. Train a TF-IDF plus logistic regression classifier
6. Evaluate the classifier against the VADER labels
7. Compare the rule-based and machine-learning outputs
8. Inspect model features and disagreements

By the end, you should be able to explain the difference between rule-based and supervised machine-learning sentiment analysis, understand why labels matter, and compare two sentiment methods on the same texts.

## Learning Objectives

1. Explain what sentiment analysis tries to measure
2. Use VADER to score text with a lexicon and rule-based method
3. Convert sentiment scores into positive, neutral, and negative labels
4. Explain why supervised machine learning requires labels
5. Train a TF-IDF text classifier with `scikit-learn`
6. Evaluate a sentiment classifier with accuracy, a classification report, and a confusion matrix
7. Compare rule-based and machine-learning sentiment results on the same dataset
8. Interpret disagreements carefully as a digital humanist


## Sentiment Analysis: An Overview

__Sentiment analysis__ is a method for estimating the emotional tone or valence of text. Many sentiment tools try to place text somewhere on a scale from negative to positive.

That sounds simple, but it gets tricky very quickly. Consider the difference between these sentences:

| sentence | why it is tricky |
| --- | --- |
| `This policy is unbelievably good.` | strong positive language |
| `This policy is unbelievably bad.` | strong negative language |
| `This policy is unbelievably good at causing confusion.` | the positive word is part of a negative idea |
| `Great. Another broken promise.` | sarcasm changes the meaning |

Sentiment analysis tools do not read like people. They work by applying rules, counting words, or learning statistical patterns from examples. Those methods can be useful, but we have to interpret the outputs carefully.

### Important Interpretive Caution

Sentiment is not the same thing as truth, opinion, political position, bias, or moral judgment.

In this notebook, we're working with newspaper article metadata and lead paragraphs. News writing often contains negative words because it reports on conflict, crisis, illness, crime, elections, lawsuits, and other public events. A negative sentiment score does not necessarily mean an article is unfair or hostile. It may simply mean the article is describing a negative event.

We also do __not__ have human-created sentiment labels for this dataset. That matters. For the machine-learning section, we will use VADER labels as __weak labels__. A weak label is an imperfect label created by a rule, heuristic, or another model. This lets us practice a supervised machine-learning workflow, but the model is learning to approximate VADER. It is not learning an objective truth about the articles.

As you work through the notebook, keep asking:

- What exactly is being scored?
- Are we measuring the tone of the writing, the event being described, or something else?
- Which texts does each method classify differently?
- What would human annotation add to this project?

## The Election Article Dataset

We'll use `election2020_articles.csv`. This is the same local dataset used in the earlier New York Times API lesson. It contains articles collected from the NYT Article Search API during the 2020 presidential election season.

Each row represents one article. The dataset includes metadata such as publication date, section, headline, abstract, and lead paragraph.

For sentiment analysis, we will use a short text field made from each article's headline plus lead paragraph. This keeps the method easy to inspect. It also lets us compare the rule-based and machine-learning approaches on exactly the same text.

## Setup

We'll use familiar libraries for data and plotting, plus two sentiment or machine-learning tools:

- `vaderSentiment` for rule-based sentiment analysis
- `scikit-learn` for TF-IDF features and machine-learning classifiers

The `vaderSentiment` package was not included in the initial class environment, so we'll install it first:

In [ ]:
%pip install vadersentiment

And here are our libraries and modules:

In [ ]:
%matplotlib inline

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


## Step 1: Load The Data

Let's read the article dataset from the course `data` folder.


In [ ]:
data_path = Path("../data/election2020_articles.csv")
articles = pd.read_csv(data_path)

articles.shape

Let's preview a few columns that will matter for this lesson.


In [ ]:
preview_columns = [
    "headline.main",
    "lead_paragraph",
    "pub_date",
    "section_name",
    "news_desk",
]

articles[preview_columns].head()

Before choosing a text column, let's check missing values in the columns we might use.


In [ ]:
articles[["headline.main", "lead_paragraph", "abstract"]].isna().sum()

## Step 2: Build A Text Column

Before we score sentiment, we have to decide what text we are actually asking the computer to read. That choice matters. Sentiment analysis does not operate on an entire article in some abstract way; it operates on whatever string of text we give it.

For this lesson, we'll use the `headline.main` and `lead_paragraph` columns. The headline is useful because headlines often contain condensed, attention-grabbing language. The lead paragraph is useful because it usually gives the article's opening context: who, what, where, and why the story matters. Together, they give us more context than the headline alone while staying short enough for us to inspect closely.

There is a tradeoff here. If we used the full article text, we might capture a richer range of tone, but the results would also be harder to interpret. If we used only the headline, the method would be easier to inspect, but it might overstate the emotional force of headline writing. The lead paragraph also has missing values in some rows, so we'll use the abstract as a backup when needed.

We'll create a new column called `analysis_text` by combining:

1. the main headline
2. the lead paragraph
3. the abstract as a backup when the lead paragraph is missing

This is a modeling choice. If we used only the headline, only the lead paragraph, or the full article text, our results would probably change.

As you look at the examples below, ask yourself: why might we want to analyze sentiment in article headlines and leads specifically? Are we measuring the emotional tone of the newspaper's framing, the events being described, or both?


In [ ]:
articles["analysis_text"] = (
    articles["headline.main"].fillna("")
    + ". "
    + articles["lead_paragraph"].fillna(articles["abstract"].fillna(""))
)

articles["analysis_text"] = articles["analysis_text"].str.replace(r"\s+", " ", regex=True).str.strip()
articles = articles[articles["analysis_text"].str.len() > 20].copy()

articles[["headline.main", "analysis_text"]].head()

Here is one complete text example. It is short enough for us to inspect, but long enough to include more context than a headline alone.


In [ ]:
articles.loc[0, "analysis_text"]

## Step 3: Rule-Based Sentiment Analysis With VADER

We'll start with the shortened version of the VADER workflow from the earlier API notebook.

VADER is a lexicon and rule-based sentiment tool. A **lexicon** is a stored list of words with associated scores. VADER also includes rules for things like punctuation, capitalization, negation, and intensifiers.

That means VADER does not learn from our dataset. It arrives with its own vocabulary and scoring rules, then applies them to each text.


In [ ]:
analyzer = SentimentIntensityAnalyzer()

example_text = articles.loc[0, "analysis_text"]
analyzer.polarity_scores(example_text)

VADER returns four scores:

- `pos`: share of the text that sounds positive
- `neu`: share of the text that sounds neutral
- `neg`: share of the text that sounds negative
- `compound`: a normalized overall score from `-1` to `1`

We'll mostly pay attention to `compound`. A score near `1` is more positive. A score near `-1` is more negative. A score near `0` is neutral.

In [ ]:
articles["vader_scores"] = articles["analysis_text"].apply(analyzer.polarity_scores)
articles["vader_compound"] = articles["vader_scores"].apply(lambda scores: scores["compound"])

articles[["headline.main", "vader_compound"]].head()

Now we'll turn the numeric VADER score into a categorical label.

VADER's common thresholds are:

- `positive`: compound score greater than or equal to `0.05`
- `neutral`: compound score between `-0.05` and `0.05`
- `negative`: compound score less than or equal to `-0.05`

These thresholds are convenient, but they are still choices. A text with a score of `0.049` and a text with a score of `0.051` are not meaningfully different just because they fall on different sides of the line.


In [ ]:
sentiment_order = ["negative", "neutral", "positive"]

def label_vader_score(score):
    if score >= 0.05:
        return "positive"
    if score <= -0.05:
        return "negative"
    return "neutral"

articles["vader_label"] = articles["vader_compound"].apply(label_vader_score)
articles["vader_label"].value_counts().reindex(sentiment_order)

A histogram gives us a quick sense of the score distribution.


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(
    data=articles,
    x="vader_compound",
    bins=24,
    color="steelblue",
)
plt.axvline(-0.05, color="darkred", linestyle="--", label="negative threshold")
plt.axvline(0.05, color="darkgreen", linestyle="--", label="positive threshold")
plt.title("VADER Compound Sentiment Scores")
plt.xlabel("Compound score")
plt.ylabel("Number of articles")
plt.legend()
plt.tight_layout()

Let's inspect the articles VADER treats as most positive.


In [ ]:
article_display_columns = [
    "headline.main",
    "vader_compound",
    "vader_label",
    "analysis_text",
]

articles.sort_values("vader_compound", ascending=False)[article_display_columns].head(5)

And now the articles VADER treats as most negative.


In [ ]:
articles.sort_values("vader_compound", ascending=True)[article_display_columns].head(5)

And let's look at some of the neutral scores.

In [ ]:
articles[articles["vader_label"] == "neutral"][
    ["vader_compound", "vader_label", "analysis_text"]
].head()

In [ ]:
articles["analysis_text"].loc[6]

### Challenge: Read The Extremes

Choose one article from the most positive list and one from the most negative list.

For each one, ask:

1. Which words seem to be driving the score?
2. Is the score describing the tone of the article or the event being described?
3. Do you agree with the label?

We can also look at average VADER sentiment over time. Because this dataset covers several weeks during the election season, weekly averages are easier to read than daily averages.


In [ ]:
articles["pub_date"] = pd.to_datetime(articles["pub_date"])

weekly_vader = articles.set_index("pub_date")["vader_compound"].resample("W").mean()

plt.figure(figsize=(10, 5))
weekly_vader.plot(marker="o", color="steelblue")
plt.axhline(0, color="black", linewidth=1)
plt.title("Average VADER Sentiment By Week")
plt.xlabel("Publication week")
plt.ylabel("Mean VADER compound score")
plt.tight_layout()


## Step 4: Prepare Labels For Machine Learning

Supervised machine learning requires labels. For example, the music-review classifier in the previous notebook used review scores to create labels. Here, our NYT dataset does not include human sentiment labels.

So what can we do?

For this tutorial, we'll use `vader_label` as a __weak label__. This means the model will learn from labels created by VADER rather than labels created by human readers.

That has a big consequence: our machine-learning classifier isn't being trained to discover true sentiment. It is being trained to predict VADER-style sentiment from text features. That can still be useful because it lets us see how a supervised model works and where it agrees or disagrees with the rule-based method. But if we wanted a more robust sentiment analysis pipeline, what could we do?

Human tag some data? How could we devise a schema for tagging sentiment?

That would take more time than we have in class, but yes, that would be an appropriate next step. But for now, let's get our Python code written with the VADER labels we already have.

We'll use the conventional machine-learning names `X` and `y`:

- `X` is the input data: the text we want the model to read.
- `y` is the target label: the category we want the model to predict.

Here, `X` will be our `analysis_text` column and `y` will be the `vader_label` column. In plain English, the task is: given the article headline and lead paragraph, can the model predict VADER's positive, neutral, or negative label?


In [ ]:
X = articles["analysis_text"]
y = articles["vader_label"]

print(f"Number of texts: {len(X)}")
print(f"Number of labels: {len(y)}")
y.value_counts().reindex(sentiment_order)

Now we'll split the data into training and test sets.

The model learns from the training set. We evaluate it on the test set, which contains articles the model did not see during training.

A few arguments in `train_test_split()` are worth slowing down for:

- `test_size=0.25` means 25 percent of the articles will go into the test set, while 75 percent will go into the training set.
- `random_state=42` makes the random split reproducible. Without it, Python could make a different split each time we rerun the notebook, which would make our results harder to compare.
- `stratify=y` tells Python to preserve roughly the same balance of positive, neutral, and negative labels in both the training and test sets.

That last setting is especially important when labels are uneven. If the test set accidentally had very few neutral articles, for example, our evaluation would give us a distorted picture of how well the model handles neutral cases.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print(f"Training articles: {len(X_train)}")
print(f"Testing articles: {len(X_test)}")

## Step 5: Establish A Baseline

Before training a real classifier, we should create a baseline. This is the same idea we used in Step 5 of `5-3_text_classification.ipynb`.

A __baseline__ is a simple point of comparison. It asks: how well could we do with a very simple rule that does not really learn from the text?

We'll use `DummyClassifier`, which comes from `sklearn.dummy`. A dummy classifier is intentionally limited. With `strategy="most_frequent"`, it checks which label appears most often in `y_train`, then predicts that label for every article in `X_test`. It does not inspect the headline, lead paragraph, words, phrases, or TF-IDF features.

That makes it a useful reality check. If our logistic regression model can't beat this baseline by a meaningful amount, then it hasn't learned much beyond the class imbalance in the training data.

This is also a good moment to connect sentiment analysis back to text classification. Sentiment analysis is one context in which we might classify texts: the labels happen to be `negative`, `neutral`, and `positive`. The workflow is still a supervised classification workflow: define labels, split data, establish a baseline, train a model, evaluate predictions, and interpret the errors. In a sense, sentiment analysis is just one form of text classification!

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline accuracy: {baseline_accuracy:.3f}")

## Step 6: Turn Text Into TF-IDF Features

Machine-learning models need numbers, not raw text. We'll use `TfidfVectorizer` to turn article text into numeric features.

At this point, TF-IDF should be familiar. We've covered it a lot in previous tutorials, especially the TF-IDF section of `week_3/3-1_text_analysis_continued.ipynb` and Step 6 of `5-3_text_classification.ipynb`. The quick reminder is that TF-IDF gives more weight to terms that are common in one document but not common across every document.

Here, each article becomes a row of numeric features. Each column is a word or two-word phrase from the learned vocabulary, and each value is that term's TF-IDF weight in that article. The classifier will use those numbers, not the raw text string itself.

We'll use settings similar to the text-classification notebook:

- `stop_words="english"` removes common English words
- `min_df=3` keeps terms that appear in at least three articles
- `max_features=5000` keeps the vocabulary manageable
- `ngram_range=(1, 2)` includes single words and two-word phrases


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    min_df=3,
    max_features=5000,
    ngram_range=(1, 2),
)

tfidf_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_test = tfidf_vectorizer.transform(X_test)

print(type(tfidf_train))
print(tfidf_train.shape)

The matrix has one row per article and one column per learned vocabulary term. Let's preview some of those terms.


In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()
feature_names[:30]

## Step 7: Train A Logistic Regression Sentiment Classifier

Now we'll train a logistic regression classifier.

For this three-label task, logistic regression learns word and phrase weights for `negative`, `neutral`, and `positive`.

We'll use a `Pipeline`, as we did in Step 7 of `5-3_text_classification.ipynb`, so the vectorizer and classifier stay connected. A pipeline chains several steps together in a fixed order. Here the first step is `TfidfVectorizer`, which turns raw article text into TF-IDF features. The second step is `LogisticRegression`, which uses those features to predict a sentiment label.

This is useful because it lets us treat the whole workflow as one model. When we call `.fit(X_train, y_train)`, the pipeline first fits the TF-IDF vectorizer on the training texts, transforms those texts into features, and then fits logistic regression on those features. When we call `.predict(X_test)`, the pipeline applies the same vectorizer to the test texts before passing them to the classifier.

Pipelines also help prevent data leakage. The vectorizer should learn its vocabulary and IDF weights from the training data only. If we accidentally fit TF-IDF on all the articles before evaluating the classifier, the test set would have influenced the feature-making step.

The `class_weight="balanced"` setting tells the model to pay extra attention to smaller classes. In this dataset, `neutral` has fewer examples than `positive` or `negative`.


In [ ]:
tfidf_settings = {
    "stop_words": "english",
    "min_df": 3,
    "max_features": 5000,
    "ngram_range": (1, 2),
}

sentiment_model = Pipeline([
    ("tfidf", TfidfVectorizer(**tfidf_settings)),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
    )),
])

sentiment_model.fit(X_train, y_train)

Now let's predict sentiment labels for the test set.


In [ ]:
ml_predictions = sentiment_model.predict(X_test)
ml_accuracy = accuracy_score(y_test, ml_predictions)

print(f"Logistic regression accuracy: {ml_accuracy:.3f}")

How did it do in terms of accuracy? How does this compare to our dummy classifier score?

We should also look at the classification report which gives us more detail than accuracy alone.

Remember: this report compares the machine-learning predictions to the VADER labels. It tells us how well the model learned to approximate those weak labels.

In [ ]:
print(classification_report(
    y_test,
    ml_predictions,
    labels=sentiment_order,
))

A confusion matrix shows where the model matched VADER and where it did not.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    ml_predictions,
    labels=sentiment_order,
    display_labels=sentiment_order,
    cmap="Blues",
    colorbar=False,
)
plt.title("ML Predictions Compared To VADER Labels")
plt.tight_layout()


### What Does This Evaluation Mean?

If the classifier improves over the baseline, it's learned some text patterns associated with the VADER labels.

But we should not overstate the result. A high score would mean the model learned to imitate VADER. A low score would mean it struggled to imitate VADER. Neither result tells us whether VADER itself is correct.

For a real sentiment-analysis research project, we would usually want at least some human-coded examples so we can evaluate the model against human interpretation.


## Step 8: Compare VADER And ML Predictions On Test Articles

Let's put the rule-based labels and machine-learning labels side by side for the test set.

Logistic regression can also estimate probabilities for each class. In simplified terms, the model calculates a weighted score for each possible label based on the TF-IDF features in an article. It then converts those scores into probabilities that add up to 1. For example, one article might be estimated as 0.20 negative, 0.30 neutral, and 0.50 positive.

We'll store the highest of those probabilities as the model's `ml_confidence`. This is not the same as being correct. It only tells us how strongly the model prefers its chosen label over the alternatives according to the patterns it learned.

Here is what the code below does:

- `predict_proba(X_test)` gets the model's estimated probability for each sentiment label in the test set.
- `pd.DataFrame(...)` turns that probability array into a labeled table, with one column per class.
- `articles.loc[X_test.index, ...]` retrieves the original article rows that ended up in the test set.
- `ml_label` stores the model's predicted label.
- `ml_confidence` stores the highest class probability for each article.
- `matches_vader` checks whether the machine-learning label matches the original VADER label.


In [ ]:
test_probabilities = sentiment_model.predict_proba(X_test)
probability_table = pd.DataFrame(
    test_probabilities,
    columns=sentiment_model.classes_,
    index=X_test.index,
)

comparison = articles.loc[X_test.index, [
    "headline.main",
    "analysis_text",
    "vader_compound",
    "vader_label",
]].copy()

comparison["ml_label"] = pd.Series(ml_predictions, index=X_test.index)
comparison["ml_confidence"] = probability_table.max(axis=1)
comparison["matches_vader"] = comparison["vader_label"] == comparison["ml_label"]

comparison.head(10)


How often do the two methods agree in the test set?

`agreement_rate` is the share of test articles where `matches_vader` is `True`.

`pd.crosstab()` creates a cross-tabulation: a table that counts how often one categorical variable appears with another categorical variable. Here, the rows are VADER labels and the columns are machine-learning labels. The diagonal cells show agreement. The off-diagonal cells show disagreements.


In [ ]:
agreement_rate = comparison["matches_vader"].mean()
print(f"Agreement rate/accuracy: {agreement_rate:.3f}")

pd.crosstab(
    comparison["vader_label"],
    comparison["ml_label"],
    rownames=["VADER label"],
    colnames=["ML label"],
).reindex(index=sentiment_order, columns=sentiment_order)


Disagreements are especially useful for interpretation. The rows below show cases where the machine-learning model and VADER label differ.


In [ ]:
disagreements = comparison[~comparison["matches_vader"]].copy()

disagreements.sort_values("ml_confidence", ascending=False)[[
    "headline.main",
    "vader_label",
    "ml_label",
    "ml_confidence",
    "vader_compound",
    "analysis_text",
]].head(10)


### Challenge: Close Read Disagreements

Choose two disagreement rows.

For each one, ask:

1. Which label seems more plausible to you: VADER or ML?
2. Which words might have pushed VADER in its direction?
3. Which words might have pushed the ML model in its direction?
4. Would a human annotator need more context than the headline and lead paragraph?


## Step 9: Whole-Dataset Exploratory Comparison

The honest evaluation happened above, when we trained on `X_train` and evaluated on `X_test`. That test set mattered because those articles were unseen during training.

Now we are going to do something different: fit a model on the full dataset so we can create whole-corpus comparison columns and visualizations. This does __not__ create a new evaluation result. Our train/test split already used the entire dataset by placing each article either in the training set or the test set. When we now call `.fit(X, y)`, the model sees every article and every VADER label.

So why do it? Because it lets us ask a different, more exploratory question: after learning from all available VADER-labeled examples, how does the machine-learning model's scoring pattern compare with VADER's scoring pattern across the whole dataset?

Read everything in this section as descriptive analysis, not proof of model accuracy. The model has seen these examples during this final fit, and the labels still come from VADER rather than human annotators.

The code below repeats the same pipeline structure, fits it on all `X` and `y`, then adds three new columns to `articles`:

- `ml_label`: the model's predicted label
- `ml_confidence`: the highest estimated class probability
- later, `ml_sentiment_score`: positive probability minus negative probability

In [ ]:
final_sentiment_model = Pipeline([
    ("tfidf", TfidfVectorizer(**tfidf_settings)),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight="balanced",
    )),
])

final_sentiment_model.fit(X, y)

articles["ml_label"] = final_sentiment_model.predict(X)

all_probabilities = final_sentiment_model.predict_proba(X)
all_probability_table = pd.DataFrame(
    all_probabilities,
    columns=final_sentiment_model.classes_,
    index=articles.index,
)

articles["ml_confidence"] = all_probability_table.max(axis=1)

articles[["headline.main", "vader_label", "ml_label", "ml_confidence"]].head()

Let's compare label counts from the rule-based method and the all-data machine-learning model.

Again, this is not an accuracy table. It simply asks whether the two methods produce similar overall distributions of negative, neutral, and positive labels when applied to the same collection of articles.


In [ ]:
label_counts = pd.DataFrame({
    "VADER": articles["vader_label"].value_counts().reindex(sentiment_order),
    "Machine learning": articles["ml_label"].value_counts().reindex(sentiment_order),
})
label_counts.index.name = "label"

label_counts

In [ ]:
label_counts_long = label_counts.reset_index().melt(
    id_vars="label",
    var_name="method",
    value_name="articles",
)

plt.figure(figsize=(8, 4))
sns.barplot(
    data=label_counts_long,
    x="label",
    y="articles",
    hue="method",
    order=sentiment_order,
)
plt.title("Sentiment Label Counts By Method")
plt.xlabel("Sentiment label")
plt.ylabel("Number of articles")
plt.tight_layout()


We can also create a rough machine-learning sentiment score by subtracting the model's negative probability from its positive probability.

This gives us a score where:

- positive values mean the model leans positive
- negative values mean the model leans negative
- values near zero mean the model is neutral, uncertain, or mixed

This score is not the same as VADER's compound score, but it lets us compare the two methods on a shared numeric scale.

The code uses the probability table created above. `all_probability_table["positive"]` selects the model's estimated positive probability for each article, and `all_probability_table["negative"]` selects the estimated negative probability. Subtracting negative from positive gives us a simple directional score.


In [ ]:
articles["ml_positive_probability"] = all_probability_table["positive"]
articles["ml_negative_probability"] = all_probability_table["negative"]
articles["ml_sentiment_score"] = (
    articles["ml_positive_probability"]
    - articles["ml_negative_probability"]
)

articles[["vader_compound", "ml_sentiment_score"]].corr()

A scatterplot helps us see where the numeric scores align and where they diverge. Points near the rising diagonal pattern suggest that VADER and the machine-learning model are ranking sentiment in similar ways. Points far from that pattern are good candidates for closer reading.


In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=articles,
    x="vader_compound",
    y="ml_sentiment_score",
    hue="vader_label",
    hue_order=sentiment_order,
    alpha=0.55,
)
plt.axhline(0, color="black", linewidth=1)
plt.axvline(0, color="black", linewidth=1)
plt.title("VADER Scores Compared To ML Scores")
plt.xlabel("VADER compound score")
plt.ylabel("ML positive probability minus negative probability")
plt.tight_layout()


And now let's compare weekly average scores from both methods.

This visualization asks whether the two scoring methods tell a similar time-based story. Because both scores are derived from the same headline/lead text, and because the machine-learning score was trained from VADER labels, we should interpret similarity cautiously. Strong agreement may mean the classifier learned to imitate VADER; it does not automatically mean either method has captured human sentiment well.


In [ ]:
weekly_scores = articles.set_index("pub_date")[[
    "vader_compound",
    "ml_sentiment_score",
]].resample("W").mean()

plt.figure(figsize=(10, 5))
weekly_scores.plot(marker="o")
plt.axhline(0, color="black", linewidth=1)
plt.title("Average Sentiment By Week: VADER And Machine Learning")
plt.xlabel("Publication week")
plt.ylabel("Average sentiment score")
plt.tight_layout()


## Step 10: Interpret Important Model Features

One reason to start with logistic regression is that we can inspect feature weights.

For each label, the model learns which terms push predictions toward that class. These terms are not explanations in a humanistic sense, but they can help us see what the model used as evidence.

Because this model was trained on VADER labels, the feature weights tell us which terms helped logistic regression reproduce VADER-style labels in this dataset. They do not tell us which words are inherently positive, neutral, or negative.

Here is what the code below does:

- `final_sentiment_model.named_steps["tfidf"]` pulls the fitted TF-IDF vectorizer out of the pipeline.
- `final_sentiment_model.named_steps["classifier"]` pulls the fitted logistic regression classifier out of the pipeline.
- `get_feature_names_out()` gives us the vocabulary terms that became TF-IDF feature columns.
- `fitted_classifier.coef_` gives us the learned model weights for those feature columns.
- `fitted_classifier.classes_` tells us the class labels in the same order as the rows of `coef_`.
- The loop goes label by label, finds that label's row of weights, sorts the weights, and keeps the 15 terms with the strongest positive weights for that label.

The resulting `important_terms` table is a compact way to see which terms most strongly pushed the model toward each sentiment label.


In [ ]:
fitted_vectorizer = final_sentiment_model.named_steps["tfidf"]
fitted_classifier = final_sentiment_model.named_steps["classifier"]

feature_names = fitted_vectorizer.get_feature_names_out()
coefficients = fitted_classifier.coef_
classes = fitted_classifier.classes_

term_rows = []

for label in sentiment_order:
    class_position = np.where(classes == label)[0][0]
    class_coefficients = coefficients[class_position]
    top_indices = np.argsort(class_coefficients)[-15:][::-1]

    for term_index in top_indices:
        term_rows.append({
            "label": label,
            "term": feature_names[term_index],
            "weight": class_coefficients[term_index],
        })

important_terms = pd.DataFrame(term_rows)
important_terms


Let's plot the top weighted terms for each class. Read these as model evidence, not as a dictionary of sentiment meanings.


In [ ]:
plot_terms = important_terms.copy()
plot_terms["term_with_label"] = plot_terms["term"] + " (" + plot_terms["label"] + ")"

plt.figure(figsize=(9, 10))
sns.barplot(
    data=plot_terms.sort_values("weight"),
    x="weight",
    y="term_with_label",
    hue="label",
    hue_order=sentiment_order,
    dodge=False,
)
plt.title("Terms With Strong Logistic Regression Weights")
plt.xlabel("Model weight")
plt.ylabel("Term")
plt.tight_layout()


### Interpreting These Terms

Some terms may look obviously emotional. Others may be names, topics, offices, places, or event words.

That is important. Because our labels came from VADER, the model may learn not only sentiment words but also topic patterns that happen to correlate with VADER's labels in this dataset.

For example, if articles about a crisis often receive negative VADER scores, the model may learn words related to that crisis. That does not mean those words are inherently negative. It means they helped the model reproduce the weak labels in this particular corpus.


## Step 11: Compare A Few Classifiers

Logistic regression is a good starting point, but it's not the only text-classification method.

Step 11 of `5-3_text_classification.ipynb` gives a fuller overview of these model types, so we won't repeat all of that explanation here. The main point is that sentiment analysis can use many of the same classification tools we use for other text-classification tasks. What changes is the meaning of the labels.

Let's compare four models using cross-validation:

- __Dummy baseline:__ always predicts the most common label
- __Naive Bayes:__ a fast classic method for text classification
- __Logistic regression:__ learns weighted evidence for each class
- __Linear SVM:__ often performs well with TF-IDF text features

We'll keep the same TF-IDF settings for each real classifier so the comparison stays simple.

In [ ]:
models = {
    "Dummy baseline": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_settings)),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]),
    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_settings)),
        ("classifier", MultinomialNB()),
    ]),
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_settings)),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight="balanced",
        )),
    ]),
    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(**tfidf_settings)),
        ("classifier", LinearSVC(
            random_state=42,
            max_iter=3000,
            class_weight="balanced",
        )),
    ]),
}


Now we'll cross-validate each model.

We'll use two metrics:

- `accuracy`: overall share of correct predictions
- `f1_macro`: average F1 score across the three labels, giving each label equal importance


In [ ]:
model_rows = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=5,
        scoring=["accuracy", "f1_macro"],
    )

    model_rows.append({
        "model": model_name,
        "mean_accuracy": scores["test_accuracy"].mean(),
        "mean_f1_macro": scores["test_f1_macro"].mean(),
        "accuracy_std": scores["test_accuracy"].std(),
    })

model_results = pd.DataFrame(model_rows).sort_values("mean_accuracy", ascending=False)
model_results

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(
    data=model_results.sort_values("mean_accuracy"),
    x="mean_accuracy",
    y="model",
    color="steelblue",
)
plt.title("Cross-Validated Accuracy By Model")
plt.xlabel("Mean accuracy")
plt.ylabel("Model")
plt.tight_layout()